# 阶段二：Colab T4 QLoRA SFT（Superseded）

> 本 notebook 仅保留历史审计证据。2026-08-08 起主路径为本机 MLX 与 `roleplay-posttrain`，不得用这里的 checkpoint 进入 GRPO。

从 Colab 菜单选择 **Runtime → Change runtime type**：Runtime Version 使用 `2026.04`，Hardware accelerator 使用单张 `T4 GPU`，然后按顺序运行全部单元。任一技术断言失败都应停止，不自动换版本、换模型或调参。

本 notebook 直接运行完整 3 epochs（约 12 个 optimizer step），训练后验证有限梯度、非零 LoRA-B 和 adapter 重载，并用 `20260807/08/09` 三个 seed 生成同后端 Base/SFT Dev 各 30 条。技术门槛与相对行为门槛通过后，主 seed 的匿名 A/B 人工复核通过才允许进入 GRPO。训练中间文件位于 `/content`；Drive 只保存约定的核心产物。


In [ ]:
# 1. 验证 Colab/T4，挂载 Drive，clone 仓库并创建独立 run。
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

try:
    import google.colab  # noqa: F401
except ImportError as exc:
    raise RuntimeError("必须在 Google Colab 托管运行时执行") from exc

import torch
from google.colab import drive

assert sys.version_info[:2] == (3, 12), platform.python_version()
assert torch.__version__.split("+")[0] == "2.10.0", torch.__version__
assert torch.cuda.is_available(), "CUDA 不可用"
assert torch.cuda.device_count() == 1, torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
assert "T4" in gpu_name, gpu_name
assert gpu_memory_gib >= 14.0, gpu_memory_gib
gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()

drive.mount("/content/drive")
REPO_URL = "https://github.com/chenkx612/local-roleplay-llm.git"
REPO_DIR = Path("/content/local-roleplay-llm")
assert not REPO_DIR.exists(), f"为避免使用旧 checkout，请重启 runtime：{REPO_DIR}"
subprocess.run(
    ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
)
repo_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
WORK_ROOT = Path("/content/roleplay-stage2-work") / RUN_ID
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/roleplay/morgana-v1/stage2-sft") / RUN_ID
WORK_ROOT.mkdir(parents=True, exist_ok=False)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=False)
CONFIG_PATH = REPO_DIR / "configs/morgana_v1_sft_t4.yaml"
assert CONFIG_PATH.is_file(), CONFIG_PATH
os.chdir(REPO_DIR)

run_summary = {
    "schema_version": 2,
    "status": "initialized",
    "run": {
        "id": RUN_ID,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "repository": REPO_URL,
        "branch": "main",
        "commit": repo_commit,
    },
    "environment": {
        "required_colab_runtime": "2026.04",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": gpu_name,
        "gpu_memory_gib": round(gpu_memory_gib, 2),
        "nvidia_smi_query": gpu_query,
    },
}
SUMMARY_PATH = DRIVE_RUN_ROOT / "run_summary.json"

def write_run_summary() -> None:
    SUMMARY_PATH.write_text(
        json.dumps(run_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )

write_run_summary()
print(json.dumps(run_summary, ensure_ascii=False, indent=2))


In [ ]:
# 2. 安装固定依赖并核对直接依赖版本。
import importlib.metadata

PINNED_PACKAGES = {
    "ms-swift": "4.4.1",
    "datasets": "4.8.4",
    "transformers": "5.12.1",
    "peft": "0.19.1",
    "bitsandbytes": "0.49.2",
    "qwen-vl-utils": "0.0.14",
    "flash-linear-attention": "0.5.1",
    "ninja": "1.13.0",
    "causal-conv1d": "1.6.2.post1",
}
assert torch.version.cuda and torch.version.cuda.startswith("12."), torch.version.cuda
assert torch._C._GLIBCXX_USE_CXX11_ABI is True

packages = [
    f"{name}=={version}"
    for name, version in PINNED_PACKAGES.items()
    if name != "causal-conv1d"
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", *packages], check=True
)
causal_version = PINNED_PACKAGES["causal-conv1d"]
causal_wheel = (
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/"
    f"v{causal_version}/causal_conv1d-{causal_version}%2B"
    "cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", causal_wheel],
    check=True,
)

installed_versions = {
    name: importlib.metadata.version(name) for name in PINNED_PACKAGES
}
normalized_versions = {
    name: version.split("+", 1)[0] for name, version in installed_versions.items()
}
assert normalized_versions == PINNED_PACKAGES, installed_versions
from datasets.features import Json
assert Json is not None

run_summary["environment"]["packages"] = installed_versions
run_summary["status"] = "environment_validated"
write_run_summary()
print(json.dumps(installed_versions, ensure_ascii=False, indent=2))


In [ ]:
# 3. 校验冻结输入、模型 revision、训练配置和模板 token 长度。
import yaml
from huggingface_hub import HfApi
from transformers import AutoProcessor

MODEL_ID = "Qwen/Qwen3.5-2B"
MODEL_REVISION = "965dcc54bc9c0591873df0e9869c056a54d323d1"
EXPECTED_INPUTS = {
    "data/runs/morgana-v1/sft_train.jsonl": {
        "records": 50,
        "sha256": "277323c097305ebbee7bfa93cb27c34247800f2dec835f6d052ede7c2b178a7a",
    },
    "data/runs/morgana-v1/dev.jsonl": {
        "records": 10,
        "sha256": "cbce0b38bb6f8b8cbef0bc45fd52b5a8212a66445569dfe0a8c7e8e88f63ddc6",
    },
}
EXPECTED_FILE_HASHES = {
    "data/runs/morgana-v1/inputs/persona.json": "8b4be4ac72b0f90ff2bd875fe318319ce1498cd11cd503e39f782ca14b46ae90",
    "data/runs/morgana-v1/system_prompt.txt": "d2cbaaa6d603b66e123c0fc435bcb8477584ea487870087d1db74a8ae4a4938a",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_jsonl(path: Path) -> list[dict]:
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        assert line.strip(), f"{path}:{line_number} 空行"
        value = json.loads(line)
        assert isinstance(value, dict), f"{path}:{line_number} 不是对象"
        rows.append(value)
    return rows

validated_inputs = {}
loaded_inputs = {}
for relative_path, expected in EXPECTED_INPUTS.items():
    path = REPO_DIR / relative_path
    rows = read_jsonl(path)
    digest = sha256_file(path)
    assert len(rows) == expected["records"], (relative_path, len(rows))
    assert digest == expected["sha256"], (relative_path, digest)
    validated_inputs[relative_path] = {"records": len(rows), "sha256": digest}
    loaded_inputs[relative_path] = rows
for relative_path, expected_digest in EXPECTED_FILE_HASHES.items():
    digest = sha256_file(REPO_DIR / relative_path)
    assert digest == expected_digest, (relative_path, digest)
    validated_inputs[relative_path] = {"sha256": digest}

sft_rows = loaded_inputs["data/runs/morgana-v1/sft_train.jsonl"]
for index, row in enumerate(sft_rows):
    assert set(row) == {"messages"}, (index, row.keys())
    messages = row["messages"]
    assert isinstance(messages, list) and len(messages) >= 3, index
    assert messages[-1]["role"] == "assistant", index
    for message in messages:
        assert set(message) == {"role", "content"}, (index, message)
        assert message["role"] in {"system", "user", "assistant"}, message
        assert isinstance(message["content"], str) and message["content"].strip()

dev_rows = loaded_inputs["data/runs/morgana-v1/dev.jsonl"]
assert len({row["id"] for row in dev_rows}) == 10
for row in dev_rows:
    assert set(row) == {"id", "scenario", "target_goals", "user"}, row.keys()
    assert all(row[key].strip() for key in ("id", "scenario", "user"))
    assert isinstance(row["target_goals"], list) and row["target_goals"]

train_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
assert train_config["model"] == MODEL_ID
assert train_config["model_revision"] == MODEL_REVISION
assert train_config["dataset"] == "data/runs/morgana-v1/sft_train.jsonl"
assert train_config["max_length"] == 1024
assert train_config["torch_dtype"] == "float32"
assert train_config["bnb_4bit_compute_dtype"] == "float32"
assert train_config["lora_dtype"] == "float32"
assert train_config["num_train_epochs"] == 3
assert train_config["save_only_model"] is True
EFFECTIVE_CONFIG_PATH = WORK_ROOT / "training_config.yaml"
EFFECTIVE_CONFIG_PATH.write_text(
    yaml.safe_dump(train_config, sort_keys=False, allow_unicode=True), encoding="utf-8"
)

model_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION)
assert model_info.sha == MODEL_REVISION, model_info.sha
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer = getattr(processor, "tokenizer", processor)
token_lengths = []
for index, row in enumerate(sft_rows):
    try:
        token_ids = processor.apply_chat_template(
            row["messages"], tokenize=True, add_generation_prompt=False, enable_thinking=False
        )
    except TypeError:
        token_ids = tokenizer.apply_chat_template(
            row["messages"], tokenize=True, add_generation_prompt=False
        )
    if isinstance(token_ids, dict):
        token_ids = token_ids["input_ids"]
    length = len(token_ids[0]) if token_ids and isinstance(token_ids[0], list) else len(token_ids)
    token_lengths.append(length)
    assert length <= train_config["max_length"], (index, length)
    answer = row["messages"][-1]["content"]
    assert answer.startswith("（") and "）" in answer, index
    assert "<think>" not in answer and "</think>" not in answer, index

system_prompt = (REPO_DIR / "data/runs/morgana-v1/system_prompt.txt").read_text(
    encoding="utf-8"
)
run_summary["model"] = {"name": MODEL_ID, "revision": MODEL_REVISION}
run_summary["inputs"] = {
    "files": validated_inputs,
    "max_sft_tokens": max(token_lengths),
    "max_length": train_config["max_length"],
}
run_summary["status"] = "inputs_validated"
write_run_summary()
print(json.dumps(run_summary["inputs"], ensure_ascii=False, indent=2))


## 完整 SFT

训练在独立子进程运行。`save_only_model` 阻止保存 optimizer 等断点状态；训练成功后仍只从临时目录挑选最终 adapter 的三个必要文件归档。


In [ ]:
# 4. 完整训练并验证梯度、LoRA 更新和 adapter 可加载性。
import numpy as np
from peft import PeftConfig
from safetensors import safe_open

TRAIN_ENV = {
    **os.environ,
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TOKENIZERS_PARALLELISM": "false",
}

def run_logged(command: list[str], log_path: Path) -> float:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log_file:
        log_file.write("COMMAND: " + subprocess.list2cmdline(command) + "\n")
        log_file.flush()
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=TRAIN_ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return time.monotonic() - started

def find_final_adapter(output_dir: Path) -> Path:
    candidates = []
    for config_path in output_dir.rglob("adapter_config.json"):
        directory = config_path.parent
        if (directory / "adapter_model.safetensors").is_file():
            match = re.search(r"checkpoint-(\d+)$", directory.name)
            step = int(match.group(1)) if match else -1
            candidates.append((step, config_path.stat().st_mtime_ns, directory))
    assert candidates, f"没有在 {output_dir} 找到 adapter"
    adapter_dir = max(candidates)[2]
    PeftConfig.from_pretrained(adapter_dir)
    return adapter_dir

def read_metric(output_dir: Path, metric: str) -> list[float]:
    candidates = []
    for path in output_dir.rglob("*.jsonl"):
        values = []
        for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            value = row.get(metric) if isinstance(row, dict) else None
            if isinstance(value, (int, float)):
                values.append(float(value))
        if values:
            candidates.append(values)
    return max(candidates, key=len, default=[])

def inspect_adapter_update(adapter_dir: Path) -> dict:
    weights_path = adapter_dir / "adapter_model.safetensors"
    tensor_count = lora_b_tensors = nonzero_lora_b_tensors = nonzero_elements = 0
    max_abs = 0.0
    with safe_open(weights_path, framework="np") as weights:
        for key in weights.keys():
            tensor = weights.get_tensor(key)
            tensor_count += 1
            assert np.isfinite(tensor).all(), f"adapter 包含非有限值: {key}"
            if tensor.size:
                max_abs = max(max_abs, float(np.abs(tensor).max()))
            if ".lora_B." in key:
                lora_b_tensors += 1
                nonzero = int(np.count_nonzero(tensor))
                nonzero_elements += nonzero
                nonzero_lora_b_tensors += int(nonzero > 0)
    result = {
        "tensor_count": tensor_count,
        "lora_b_tensors": lora_b_tensors,
        "nonzero_lora_b_tensors": nonzero_lora_b_tensors,
        "nonzero_lora_b_elements": nonzero_elements,
        "max_abs": max_abs,
    }
    assert lora_b_tensors > 0, result
    assert nonzero_lora_b_tensors == lora_b_tensors, result
    assert nonzero_elements > 0, result
    return result

TRAIN_DIR = WORK_ROOT / "full"
TRAIN_LOG = WORK_ROOT / "train.log"
train_command = [
    "swift",
    "sft",
    str(EFFECTIVE_CONFIG_PATH),
    "--output_dir",
    str(TRAIN_DIR),
    "--add_version",
    "false",
]
try:
    train_duration_seconds = run_logged(train_command, TRAIN_LOG)
except BaseException as exc:
    run_summary["status"] = "training_failed"
    run_summary["error"] = repr(exc)
    if TRAIN_LOG.exists():
        shutil.copy2(TRAIN_LOG, DRIVE_RUN_ROOT / "train.log")
    write_run_summary()
    raise
shutil.copy2(TRAIN_LOG, DRIVE_RUN_ROOT / "train.log")

FULL_ADAPTER = find_final_adapter(TRAIN_DIR)
full_losses = read_metric(TRAIN_DIR, "loss")
full_grad_norms = read_metric(TRAIN_DIR, "grad_norm")
expected_optimizer_steps = (
    math.ceil(len(sft_rows) / train_config["gradient_accumulation_steps"])
    * train_config["num_train_epochs"]
)
assert full_losses and all(math.isfinite(value) and value > 0 for value in full_losses), full_losses
assert full_grad_norms and all(
    math.isfinite(value) and value > 0 for value in full_grad_norms
), full_grad_norms
assert len(full_grad_norms) == expected_optimizer_steps, (
    len(full_grad_norms), expected_optimizer_steps
)
adapter_update = inspect_adapter_update(FULL_ADAPTER)
adapter_weights = FULL_ADAPTER / "adapter_model.safetensors"
run_summary["training"] = {
    "command": train_command,
    "epochs": train_config["num_train_epochs"],
    "expected_optimizer_steps": expected_optimizer_steps,
    "optimizer_steps": len(full_grad_norms),
    "duration_seconds": round(train_duration_seconds, 3),
    "losses": full_losses,
    "grad_norms": full_grad_norms,
    "adapter_update": adapter_update,
    "adapter_sha256": sha256_file(adapter_weights),
    "adapter_archive": "adapter",
}
run_summary["status"] = "training_validated"
write_run_summary()
print(json.dumps(run_summary["training"], ensure_ascii=False, indent=2))


In [ ]:
# 5. 三个 seed 分别重置 RNG，生成同一 HF 后端的 Base/SFT Dev。
from peft import PeftModel
from roleplay.sft_eval import (
    EVALUATION_SEEDS,
    PRIMARY_EVALUATION_SEED,
    build_manual_review,
    empty_manual_review_results,
    evaluate_manual_review,
    evaluate_relative_behavior_gate,
    normalize_empty_think_wrapper,
)
from swift import InferRequest, RequestConfig, TransformersEngine, get_model_processor, get_template
from transformers import BitsAndBytesConfig

request_config = RequestConfig(
    max_tokens=256,
    temperature=0.6,
    top_p=0.8,
    top_k=20,
    repetition_penalty=1.45,
)
requests = [
    InferRequest(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row["user"]},
        ]
    )
    for row in dev_rows
]
expected_ids = [row["id"] for row in dev_rows]

def reset_generation_rng(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def records_from_responses(responses, seed: int) -> list[dict]:
    assert len(responses) == len(dev_rows) == 10
    records = []
    for source, response in zip(dev_rows, responses, strict=True):
        choice = response.choices[0]
        raw_assistant = choice.message.content
        assert isinstance(raw_assistant, str) and raw_assistant.strip(), source["id"]
        assistant = normalize_empty_think_wrapper(raw_assistant)
        assert assistant, source["id"]
        records.append(
            {
                "seed": seed,
                "id": source["id"],
                "scenario": source["scenario"],
                "target_goals": source["target_goals"],
                "user": source["user"],
                "assistant": assistant,
                "raw_assistant": raw_assistant,
                "finish_reason": choice.finish_reason,
                "attempts": 1,
            }
        )
    return records

def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("x", encoding="utf-8") as output_file:
        for row in rows:
            output_file.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json_artifact(path: Path, value: dict) -> None:
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )

try:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float32,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model, processor = get_model_processor(
        MODEL_ID,
        revision=MODEL_REVISION,
        torch_dtype=torch.float32,
        quantization_config=quantization_config,
        use_hf=True,
    )
    base_engine = TransformersEngine(
        model, template=get_template(processor, enable_thinking=False)
    )
    base_records = []
    for seed in EVALUATION_SEEDS:
        reset_generation_rng(seed)
        responses = base_engine.infer(requests, request_config=request_config)
        base_records.extend(records_from_responses(responses, seed))
    del base_engine
    gc.collect()
    torch.cuda.empty_cache()

    model = PeftModel.from_pretrained(model, FULL_ADAPTER)
    sft_engine = TransformersEngine(
        model, template=get_template(processor, enable_thinking=False)
    )
    sft_records = []
    for seed in EVALUATION_SEEDS:
        reset_generation_rng(seed)
        responses = sft_engine.infer(requests, request_config=request_config)
        sft_records.extend(records_from_responses(responses, seed))
    del sft_engine, model, processor
    gc.collect()
    torch.cuda.empty_cache()

    relative_behavior_gate = evaluate_relative_behavior_gate(
        base_records, sft_records, EVALUATION_SEEDS, expected_ids
    )
    manual_packet, manual_answer_key = build_manual_review(
        base_records, sft_records, expected_ids
    )
    manual_results = empty_manual_review_results(manual_packet)
    HF_BASE_OUTPUTS = WORK_ROOT / "hf_base_dev_outputs.jsonl"
    DEV_OUTPUTS = WORK_ROOT / "dev_outputs.jsonl"
    MANUAL_PACKET = WORK_ROOT / "manual_review_packet.json"
    MANUAL_ANSWER_KEY = WORK_ROOT / "manual_review_answer_key.json"
    MANUAL_RESULTS = WORK_ROOT / "manual_review_results.json"
    write_jsonl(HF_BASE_OUTPUTS, base_records)
    write_jsonl(DEV_OUTPUTS, sft_records)
    write_json_artifact(MANUAL_PACKET, manual_packet)
    write_json_artifact(MANUAL_ANSWER_KEY, manual_answer_key)
    write_json_artifact(MANUAL_RESULTS, manual_results)
except BaseException as exc:
    run_summary["status"] = "inference_or_evaluation_failed"
    run_summary["error"] = repr(exc)
    write_run_summary()
    raise

base_metrics = relative_behavior_gate["base"]
sft_metrics = relative_behavior_gate["sft"]
technical_gate = {
    "training_gradients_finite": all(
        math.isfinite(value) and value > 0 for value in full_grad_norms
    ),
    "optimizer_step_count_expected": (
        len(full_grad_norms) == expected_optimizer_steps
    ),
    "adapter_updated": (
        adapter_update["nonzero_lora_b_tensors"] == adapter_update["lora_b_tensors"]
    ),
    "adapter_reloaded_and_generated": sft_metrics["overall"]["nonempty_count"] == 30,
    "base_dev_complete": base_metrics["overall"]["nonempty_count"] == 30,
    "sft_dev_complete": sft_metrics["overall"]["nonempty_count"] == 30,
    "outputs_aligned": relative_behavior_gate["checks"]["complete_and_aligned"],
}
run_summary["evaluation_seeds"] = list(EVALUATION_SEEDS)
run_summary["inference"] = {
    "backend": "ms-swift TransformersEngine",
    "generation": {
        "max_tokens": 256,
        "temperature": 0.6,
        "top_p": 0.8,
        "top_k": 20,
        "repetition_penalty": 1.45,
        "enable_thinking": False,
    },
    "rng_reset": "Base 和 SFT 在每个 seed 推理前分别重置 Python/Torch/CUDA RNG",
    "normalization": "仅移除开头的空 <think></think> wrapper；保留 raw_assistant",
    "stage1_backend_difference": (
        "TransformersEngine 不支持 Stage 1 MLX Base 的 presence_penalty=0.4"
    ),
    "hf_base": {
        **base_metrics,
        "file": HF_BASE_OUTPUTS.name,
        "sha256": sha256_file(HF_BASE_OUTPUTS),
    },
    "sft": {
        **sft_metrics,
        "file": DEV_OUTPUTS.name,
        "sha256": sha256_file(DEV_OUTPUTS),
    },
}
run_summary["technical_gate"] = technical_gate
run_summary["technically_valid"] = all(technical_gate.values())
run_summary["relative_behavior_gate"] = relative_behavior_gate
run_summary["manual_review"] = {
    "status": (
        "awaiting_manual_review"
        if relative_behavior_gate["passed"] else "not_started_behavior_failed"
    ),
    "primary_seed": PRIMARY_EVALUATION_SEED,
    "packet": MANUAL_PACKET.name,
    "answer_key": MANUAL_ANSWER_KEY.name,
    "results": MANUAL_RESULTS.name,
}
run_summary["ready_for_grpo"] = False
if not run_summary["technically_valid"]:
    run_summary["status"] = "technical_validation_failed"
    write_run_summary()
    raise AssertionError(technical_gate)
run_summary["status"] = (
    "awaiting_manual_review"
    if relative_behavior_gate["passed"] else "behavior_failed"
)
write_run_summary()
print(json.dumps(run_summary["relative_behavior_gate"], ensure_ascii=False, indent=2))


In [ ]:
# 6. 只归档核心文件，验证目录契约后删除 Colab 临时训练目录。
shutil.copy2(EFFECTIVE_CONFIG_PATH, DRIVE_RUN_ROOT / "training_config.yaml")
shutil.copy2(HF_BASE_OUTPUTS, DRIVE_RUN_ROOT / HF_BASE_OUTPUTS.name)
shutil.copy2(DEV_OUTPUTS, DRIVE_RUN_ROOT / DEV_OUTPUTS.name)
shutil.copy2(MANUAL_PACKET, DRIVE_RUN_ROOT / MANUAL_PACKET.name)
shutil.copy2(MANUAL_ANSWER_KEY, DRIVE_RUN_ROOT / MANUAL_ANSWER_KEY.name)
shutil.copy2(MANUAL_RESULTS, DRIVE_RUN_ROOT / MANUAL_RESULTS.name)

adapter_archive = DRIVE_RUN_ROOT / "adapter"
adapter_archive.mkdir()
adapter_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "additional_config.json",
]
for name in adapter_files:
    source = FULL_ADAPTER / name
    assert source.is_file(), source
    shutil.copy2(source, adapter_archive / name)

artifact_paths = [
    DRIVE_RUN_ROOT / "training_config.yaml",
    DRIVE_RUN_ROOT / "train.log",
    *(adapter_archive / name for name in adapter_files),
    DRIVE_RUN_ROOT / HF_BASE_OUTPUTS.name,
    DRIVE_RUN_ROOT / DEV_OUTPUTS.name,
    DRIVE_RUN_ROOT / MANUAL_PACKET.name,
    DRIVE_RUN_ROOT / MANUAL_ANSWER_KEY.name,
    DRIVE_RUN_ROOT / MANUAL_RESULTS.name,
]
run_summary["artifacts"] = {
    str(path.relative_to(DRIVE_RUN_ROOT)): {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
}
run_summary["archived_at_utc"] = datetime.now(timezone.utc).isoformat()
run_summary.pop("error", None)
write_run_summary()

expected_files = {
    "run_summary.json",
    "training_config.yaml",
    "train.log",
    "adapter/adapter_model.safetensors",
    "adapter/adapter_config.json",
    "adapter/additional_config.json",
    "hf_base_dev_outputs.jsonl",
    "dev_outputs.jsonl",
    "manual_review_packet.json",
    "manual_review_answer_key.json",
    "manual_review_results.json",
}
actual_files = {
    str(path.relative_to(DRIVE_RUN_ROOT))
    for path in DRIVE_RUN_ROOT.rglob("*")
    if path.is_file()
}
assert actual_files == expected_files, {
    "missing": sorted(expected_files - actual_files),
    "unexpected": sorted(actual_files - expected_files),
}
shutil.rmtree(WORK_ROOT)
print(f"Stage 2 核心产物已归档：{DRIVE_RUN_ROOT}")
if run_summary["status"] == "behavior_failed":
    print("自动相对行为门槛失败：产物已保留，但禁止进入 GRPO。")
else:
    print("请匿名复核主 seed 的 10 对输出，填写 manual_review_results.json 后运行下一单元。")


In [ ]:
# 7. 人工填写结果文件后重新运行本单元，完成最终 GRPO 决策。
manual_results_path = DRIVE_RUN_ROOT / "manual_review_results.json"
submitted = json.loads(manual_results_path.read_text(encoding="utf-8"))
if not submitted.get("results"):
    print("尚未提交人工复核；run_summary 保持 awaiting_manual_review 或 behavior_failed。")
else:
    archived_packet = json.loads(
        (DRIVE_RUN_ROOT / "manual_review_packet.json").read_text(encoding="utf-8")
    )
    archived_answer_key = json.loads(
        (DRIVE_RUN_ROOT / "manual_review_answer_key.json").read_text(encoding="utf-8")
    )
    manual_gate = evaluate_manual_review(
        archived_packet, archived_answer_key, submitted
    )
    run_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    run_summary["manual_review"] = {
        **run_summary["manual_review"],
        "status": "passed" if manual_gate["passed"] else "failed",
        "gate": manual_gate,
    }
    run_summary["ready_for_grpo"] = bool(
        run_summary["technically_valid"]
        and run_summary["relative_behavior_gate"]["passed"]
        and manual_gate["passed"]
    )
    if not run_summary["relative_behavior_gate"]["passed"]:
        run_summary["status"] = "behavior_failed"
    else:
        run_summary["status"] = (
            "ready_for_grpo" if run_summary["ready_for_grpo"] else "manual_failed"
        )
    result_artifact = run_summary["artifacts"]["manual_review_results.json"]
    result_artifact["bytes"] = manual_results_path.stat().st_size
    result_artifact["sha256"] = sha256_file(manual_results_path)
    run_summary["manual_reviewed_at_utc"] = datetime.now(timezone.utc).isoformat()
    write_run_summary()
    print(json.dumps(run_summary["manual_review"], ensure_ascii=False, indent=2))
    print(f"ready_for_grpo={run_summary['ready_for_grpo']}")
